# F15 — Empaquetado de shards DL sobre Colab (cómputo fuera del PC local)

Política del equipo (13-oct): el cómputo pesado corre en Colab; el repo local
solo recibe los artefactos ligeros ya verificados. Este notebook provisiona
el dato crudo ZZU (~4 GB comprimido), ejecuta `scripts/empaquetar_shards_dl.py`
(int16, shards ~1 GB, splits 80/10/10 por paciente, stats de train) y sube
los shards a Drive para F16–F18.

Las celdas 1–2 (SSH) permiten que el asistente conduzca el resto por SSH
(`ssh colab-f15`); si prefieres la UI de Colab, ejecuta de arriba a abajo.

## 1. Setup SSH (igual que colab_f14)

In [ ]:
%%bash
apt-get -qq update >/dev/null && apt-get -qq install -y openssh-server p7zip-full >/dev/null
mkdir -p /run/sshd /root/.ssh && chmod 700 /root/.ssh
sed -i 's/#\?Port .*/Port 22/' /etc/ssh/sshd_config
nohup /usr/sbin/sshd >/dev/null 2>&1 &
echo "sshd arriba"

In [ ]:
PUBKEY = "ssh-ed25519 AAAA... tu@pc"
TS_AUTHKEY = "tskey-auth-..."

open("/root/.ssh/authorized_keys", "w").write(PUBKEY + "\n")
import os; os.chmod("/root/.ssh/authorized_keys", 0o600)
!curl -fsSL https://tailscale.com/install.sh | sh
!tailscale up --hostname colab-f15 --authkey="$TS_AUTHKEY" --ssh=false
!tailscale ip -4

## 2. Repo + dependencias + dato crudo ZZU

In [ ]:
REPO = "https://codeberg.org/norlek/Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG.git"
%cd /content
!test -d Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG || git clone --depth 1 "$REPO" -b etapa2-corpus-combinado
%cd Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG
!git fetch origin && git reset --hard origin/etapa2-corpus-combinado
!git log --oneline -1
!pip install -q wfdb pywavelets
import wfdb, pywt, sklearn, pandas, numpy
print('deps OK')

In [ ]:
# ZZU pECG multipartito con verificación MD5 (docs/metodologia_zzu_pecg.md §1)
ARCHIVOS = {
    'Child_ecg.z01': ('https://ndownloader.figshare.com/files/53524583', '4782e8c17b478745853db85210ce46aa'),
    'Child_ecg.zip': ('https://ndownloader.figshare.com/files/53523902', 'd7bb207d0f559c5f697bd7e442975718'),
}
import hashlib, pathlib, subprocess
dl = pathlib.Path('/content/descargas'); dl.mkdir(exist_ok=True)
for nombre, (url, md5) in ARCHIVOS.items():
    destino = dl / nombre
    if not destino.exists():
        subprocess.run(['curl', '-L', '-C', '-', '-o', str(destino), url], check=True)
    h = hashlib.md5(destino.read_bytes()).hexdigest()
    assert h == md5, f'{nombre}: MD5 {h} != {md5}'
    print(nombre, 'MD5 OK')
data = pathlib.Path('data/zzu_pecg'); data.mkdir(parents=True, exist_ok=True)
if not (data / 'Child_ecg').exists():
    subprocess.run(['7z', 'x', f'-o{data}', str(dl / 'Child_ecg.zip')], check=True)
n_hea = len(list((data / 'Child_ecg').rglob('*.hea')))
assert n_hea == 14190, n_hea
print(f'Child_ecg: {n_hea} registros')

## 3. F15 — empaquetado + verificación (~40–60 min en runtime CPU)

In [ ]:
!python scripts/empaquetar_shards_dl.py
!cat results/dl/verificacion_shards.txt

## 4. Persistencia: shards a Drive, manifiesto a git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/arritmias_dl/shards
!cp -n results/dl/shards/*.npy /content/drive/MyDrive/arritmias_dl/shards/
!cp results/dl/{manifiesto,splits}.csv results/dl/{normalizacion.json,verificacion_shards.txt} /content/drive/MyDrive/arritmias_dl/
!ls -lh /content/drive/MyDrive/arritmias_dl/shards | head
# Artefactos ligeros al repo (rama etapa2-corpus-combinado):
!git config user.email "colab@runtime" && git config user.name "colab"
!git add results/dl/manifiesto.csv results/dl/splits.csv results/dl/normalizacion.json results/dl/verificacion_shards.txt
!git commit -m "F15: shards DL empaquetados en Colab (verificacion VERDE)" && git push origin etapa2-corpus-combinado
# (push desde Colab requiere token: si falla, el asistente commitea desde local)